In [1]:
import os
import pandas as pd
from rapidfuzz import process, fuzz

# 1. Detección automática de la carpeta
if os.path.exists('../data/exercises_limpio.csv'):
    ruta_data = '../data/'
elif os.path.exists('data/exercises_limpio.csv'):
    ruta_data = 'data/'
else:
    raise FileNotFoundError("❌ No encuentro exercises_limpio.csv. Corre primero el notebook 01_preparacion_eda.ipynb completo.")

print(f"✅ Ruta correcta detectada: {ruta_data}")

# 2. Cargar el CSV YA LIMPIO
print("Cargando exercises_limpio.csv...")
df_base = pd.read_csv(ruta_data + 'exercises_limpio.csv')
df_base['name_clean'] = df_base['name'].str.lower().str.strip()

# 3. Cargar el dataset externo de Kaggle
df_mega = pd.read_csv(ruta_data + 'megaGymDataset.csv')
df_mega['Title_clean'] = df_mega['Title'].str.lower().str.strip()
opciones_mega = df_mega['Title_clean'].tolist()

# 4. Cruce difuso (Fuzzy Matching)
print("Calculando cruce difuso (esto tomará unos segundos)...")
def buscar_mejor_coincidencia(query):
    match = process.extractOne(query, opciones_mega, scorer=fuzz.WRatio)
    return (match[0], match[1]) if match and match[1] >= 80 else (None, 0)

resultados = df_base['name_clean'].apply(buscar_mejor_coincidencia)
df_base['best_match'] = [res[0] for res in resultados]

cobertura = (df_base['best_match'].notna().sum() / len(df_base)) * 100
print(f"\n---> Porcentaje real de cobertura del match: {cobertura:.2f}% <---")

# 5. Unir y guardar el dataset final
df_enriched = pd.merge(df_base, df_mega[['Title_clean', 'Type', 'Level', 'Rating']], left_on='best_match', right_on='Title_clean', how='left')
df_enriched = df_enriched.drop(columns=['name_clean', 'Title_clean', 'best_match'])
df_enriched[['Type', 'Level', 'Rating']] = df_enriched[['Type', 'Level', 'Rating']].fillna('No especificado')

ruta_final = ruta_data + 'exercise_features_enriched.csv'
df_enriched.to_csv(ruta_final, index=False)

print(f"\n¡Éxito total! Archivo enriquecido guardado en: {ruta_final}")
print(f"Columnas finales: {df_enriched.columns.tolist()}")

✅ Ruta correcta detectada: ../data/
Cargando exercises_limpio.csv...
Calculando cruce difuso (esto tomará unos segundos)...

---> Porcentaje real de cobertura del match: 99.62% <---

¡Éxito total! Archivo enriquecido guardado en: ../data/exercise_features_enriched.csv
Columnas finales: ['id', 'name', 'category', 'body_part', 'equipment', 'instructions', 'instruction_steps', 'muscle_group', 'secondary_muscles', 'target', 'image', 'gif_url', 'media_id', 'created_at', 'attribution', 'num_instrucciones', 'num_musculos_involucrados', 'instrucciones_texto_es', 'musculo_primario', 'Type', 'Level', 'Rating']


In [2]:
print(df_enriched.columns.tolist())

['id', 'name', 'category', 'body_part', 'equipment', 'instructions', 'instruction_steps', 'muscle_group', 'secondary_muscles', 'target', 'image', 'gif_url', 'media_id', 'created_at', 'attribution', 'num_instrucciones', 'num_musculos_involucrados', 'instrucciones_texto_es', 'musculo_primario', 'Type', 'Level', 'Rating']
